# Ball Tracking on Colab GPU (no GitHub)

Runs the ball detector on a free **NVIDIA GPU** (Colab's T4). The tracker code is embedded below (cell 5) — nothing to clone or upload. Uses the original detection logic (conf 0.65 + motion mask), writes the `interpolated` column, and can ALSO produce the annotated `.mp4` like the local script.

**Setup:** put `ball_tracker.pt` and your video in Google Drive under **`MyDrive/Models/`**.

**Steps:** Runtime -> Change runtime type -> **GPU**, then Runtime -> **Run all**. The CSV (and, if enabled, the annotated video) download at the end; drop the CSV into your PC's `outputs/ball_coordinates/`.

## 1. Check the GPU
If `CUDA available: False`, set Runtime -> Change runtime type -> GPU, then Run all again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU - enable it (Runtime > Change runtime type > GPU) and Run all again.')

## 2. Install ultralytics

In [ ]:
!pip -q install ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)

## 3. Mount Google Drive
Approve the popup. Your files live in `MyDrive/Models/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Pick your model + video, and choose whether to make the video
Both files expected in **`MyDrive/Models/`**. Set `MAKE_VIDEO = True` to also produce the annotated `.mp4` (slower: a second pass over the frames). Set it `False` for CSV-only (fast).

In [ ]:
import os
MODELS_DIR = '/content/drive/MyDrive/Models'
MODEL_NAME = 'ball_tracker.pt'        # <-- your weights filename
VIDEO_NAME = 'Input_video2.mp4'       # <-- your video filename
MAKE_VIDEO = True                     # <-- True = also write annotated mp4

MODEL_PATH = os.path.join(MODELS_DIR, MODEL_NAME)
VIDEO_PATH = os.path.join(MODELS_DIR, VIDEO_NAME)
print('Files in', MODELS_DIR, ':')
print(' ', os.listdir(MODELS_DIR) if os.path.isdir(MODELS_DIR) else 'FOLDER NOT FOUND')
assert os.path.exists(MODEL_PATH), f'Model not found: {MODEL_PATH}'
assert os.path.exists(VIDEO_PATH), f'Video not found: {VIDEO_PATH}'
print('OK  model:', round(os.path.getsize(MODEL_PATH)/1e6,1), 'MB  | video:', VIDEO_NAME,
      '| make video:', MAKE_VIDEO)

## 5. Ball tracker code  (paste target)
Trimmed, headless tracker — same detection as your local `BallTracking.py`, writes the `interpolated` column, and has the annotated-video second pass (without `cv2.imshow`). If you change the local file later, replace everything **below the `%%writefile` line** with the new constants + `BallTracker` class; `run()` here takes `(video_path, csv_path, output_path=None)`.

In [ ]:
%%writefile ball_tracker_colab.py
import csv
import os

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO


# Minimum YOLO confidence for a ball detection to be accepted.
BALL_CONF = 0.65
MOTION_THRESH = 15


class BallTracker:
    def __init__(self, model_path):
        self.model = YOLO(model_path)

    def _has_motion(self, prev_gray, curr_gray, x1, y1, x2, y2, threshold=0.05):
        "True if at least threshold fraction of pixels in the box moved."
        H, W = curr_gray.shape[:2]
        x1 = max(0, min(int(x1), W))
        x2 = max(0, min(int(x2), W))
        y1 = max(0, min(int(y1), H))
        y2 = max(0, min(int(y2), H))
        if x2 <= x1 or y2 <= y1:
            return False
        prev_slice = prev_gray[y1:y2, x1:x2]
        curr_slice = curr_gray[y1:y2, x1:x2]
        if prev_slice.size == 0 or curr_slice.size == 0:
            return False
        motion = cv2.absdiff(prev_slice, curr_slice)
        _, binary = cv2.threshold(motion, MOTION_THRESH, 255, cv2.THRESH_BINARY)
        total = binary.size
        if total == 0:
            return False
        return (binary.sum() / 255) / total >= threshold

    def detect_frame(self, frame, prev_gray=None):
        "Highest-confidence detection that also shows motion, or {} if none."
        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        results = self.model.predict(frame, conf=BALL_CONF, verbose=False)[0]
        for idx in results.boxes.conf.argsort(descending=True):
            x1, y1, x2, y2 = map(int, results.boxes.xyxy[idx])
            if prev_gray is not None and not self._has_motion(
                    prev_gray, curr_gray, x1, y1, x2, y2):
                continue
            return {1: [x1, y1, x2, y2]}, curr_gray
        return {}, curr_gray

    def draw_bboxes(self, frames, ball_positions):
        "Draws ball bounding boxes on copies of the input frames."
        output = []
        for frame, pos in zip(frames, ball_positions):
            out = frame.copy()
            bbox = pos.get(1)
            if bbox and not np.any(np.isnan(bbox)):
                x1, y1, x2, y2 = map(int, bbox)
                cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(out, "Ball", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            output.append(out)
        return output

    def interpolate_ball_positions(self, ball_positions):
        "Fill gaps; also return real_mask (True = genuine detection, not a fill)."
        real_mask = [len(p.get(1, [])) == 4 for p in ball_positions]
        positions_list = [p.get(1, []) for p in ball_positions]
        df = pd.DataFrame(positions_list, columns=["x1", "y1", "x2", "y2"])
        df = df.interpolate(limit_direction="both")
        df = df.bfill().ffill()
        return [{1: row} for row in df.to_numpy().tolist()], real_mask

    def _write_csv(self, csv_path, ball_positions, real_mask=None):
        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["frame", "x", "y", "w", "h", "cx", "cy", "area",
                             "interpolated"])
            for frame_idx, pos in enumerate(ball_positions):
                bbox = pos.get(1)
                if bbox is None or np.any(np.isnan(bbox)):
                    continue
                x1, y1, x2, y2 = bbox
                x, y = int(x1), int(y1)
                w, h = int(x2 - x1), int(y2 - y1)
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                area = w * h
                is_real = real_mask[frame_idx] if real_mask is not None else True
                writer.writerow([frame_idx, x, y, w, h, cx, cy, area,
                                 0 if is_real else 1])

    def run(self, video_path, csv_path, output_path=None):
        "Pass 1: detect+interpolate -> CSV. Pass 2 (if output_path): draw boxes -> mp4."
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print("Cannot open video:", video_path)
            return
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not fps or not np.isfinite(fps) or fps <= 0:
            fps = 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"Detecting ball in {total or '?'} frames...")
        ball_positions = []
        prev_gray = None
        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                detection, prev_gray = self.detect_frame(frame, prev_gray)
                ball_positions.append(detection)
        finally:
            cap.release()
        ball_positions, real_mask = self.interpolate_ball_positions(ball_positions)
        os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
        self._write_csv(csv_path, ball_positions, real_mask)
        print(f"CSV saved to {csv_path}")

        if not output_path:
            return
        # Pass 2: re-read and draw the (interpolated) boxes one frame at a
        # time, writing on the fly. No cv2.imshow (Colab has no GUI window).
        cap = cv2.VideoCapture(video_path)
        writer = None
        idx = 0
        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                pos = ball_positions[idx] if idx < len(ball_positions) else {}
                out = self.draw_bboxes([frame], [pos])[0]
                if writer is None:
                    h, w = out.shape[:2]
                    writer = cv2.VideoWriter(
                        output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
                writer.write(out)
                idx += 1
        finally:
            cap.release()
            if writer is not None:
                writer.release()
                print(f"Annotated video saved to {output_path}")

## 6. Run detection on the GPU
Pass 1 (detection -> CSV) runs on the GPU. Pass 2 (drawing the annotated mp4) is CPU and only runs if `MAKE_VIDEO` is True.

In [ ]:
import os, time, importlib
import ball_tracker_colab; importlib.reload(ball_tracker_colab)
from ball_tracker_colab import BallTracker

video_stem = os.path.splitext(VIDEO_NAME)[0]
OUT_CSV = f'/content/ball_{video_stem}.csv'
OUT_MP4 = f'/content/ball_tracking_{video_stem}.mp4' if MAKE_VIDEO else None

t0 = time.time()
tracker = BallTracker(MODEL_PATH)
tracker.run(VIDEO_PATH, OUT_CSV, output_path=OUT_MP4)
print(f'Elapsed: {time.time()-t0:.1f} s')

## 7. Sanity-check the CSV
Header must end with `...,area,interpolated` (`interpolated==1` = an interpolated fill, not a real detection).

In [ ]:
import pandas as pd
df = pd.read_csv(OUT_CSV)
print('columns:', list(df.columns))
print('rows:', len(df), '| frame range:', int(df.frame.min()), '->', int(df.frame.max()))
print('interpolated frames:', int(df['interpolated'].sum()),
      '/ real:', int((df['interpolated']==0).sum()))
df.head()

## 8. (Optional) Preview the annotated video inline
OpenCV's `mp4v` codec doesn't always play inline in the browser; we re-encode to H.264 with ffmpeg for a reliable preview. Skips cleanly if `MAKE_VIDEO` was False.

In [ ]:
if MAKE_VIDEO and OUT_MP4 and os.path.exists(OUT_MP4):
    PREVIEW = '/content/preview_h264.mp4'
    !ffmpeg -y -loglevel error -i "$OUT_MP4" -vcodec libx264 -pix_fmt yuv420p "$PREVIEW"
    from IPython.display import HTML
    from base64 import b64encode
    data = b64encode(open(PREVIEW, 'rb').read()).decode()
    display(HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))
else:
    print('No annotated video (MAKE_VIDEO is False or file missing).')

## 9. Save outputs
Copies the CSV (and video, if made) to `MyDrive/Models/` and downloads them. On your PC, put the CSV in `outputs/ball_coordinates/` and re-run shot analysis.

In [ ]:
import shutil
from google.colab import files
shutil.copy(OUT_CSV, MODELS_DIR)
print('Copied CSV to', os.path.join(MODELS_DIR, os.path.basename(OUT_CSV)))
files.download(OUT_CSV)
if MAKE_VIDEO and OUT_MP4 and os.path.exists(OUT_MP4):
    shutil.copy(OUT_MP4, MODELS_DIR)
    print('Copied video to', os.path.join(MODELS_DIR, os.path.basename(OUT_MP4)))
    files.download(OUT_MP4)